# Mini Food Delivery Assistant

**AI Fundamentals (AIF1303) · Final Project · Topic 20**

A courier delivers food across a 6×6 neighbourhood. A traffic alert comes in for the bridge on the planned route, either before departure or while the courier is already driving.

| Step | Question | Algorithm |
|---|---|---|
| 1 | What is the shortest route? | A* search (compared with BFS) |
| 2 | How likely is the bridge blocked? | Bayesian inference |
| 3 | Continue or take the detour? | MDP, one Bellman update (checked against full value iteration) |
| 4 | Is the assistant actually better? | Evaluation by simulation |
| 5 | What if the alert arrives on the way? | All three again, from the courier's position |
| 6 | Where does the decision flip? | Decision map |
| 7 | What really happens on the road? | Trip simulation with the true bridge state |
| 8 | Does it scale? | The same algorithms on 10×10 and 14×14 maps |

The steps feed each other: A* gives the route **and** the detour length, the detour length sets the reward for rerouting, and the Bayes posterior is the belief the MDP decision is weighted by.

All numbers are assumptions for the demonstration, as the project guidelines allow. Run the cells in order.

## Setup

The algorithms live in the `delivery` package of the repository. In Google Colab the next cell clones the repository first; run locally from the repository folder, it just imports the package.

To try other scenarios, change the numbers in `delivery/config.py` or switch the map with `use_map('10x10')`.

In [ ]:
import os, sys

if not os.path.isdir('delivery'):
    # Google Colab: fetch the project first
    if not os.path.isdir('AI-Final'):
        os.system('git clone -q https://github.com/brokehustler1237/AI-Final.git')
    os.chdir('AI-Final')
sys.path.insert(0, os.getcwd())

from delivery import *
from delivery import world
from delivery.config import *
from delivery.report import (run_astar, run_bayes, run_mdp, run_evaluation,
                             run_on_the_way, run_policy_map)

print('map:', world.MAP_NAME, '| start', world.START, '| customer', world.GOAL, '| bridge', world.ALERT_CELL)

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
from matplotlib.patches import FancyBboxPatch
from matplotlib.colors import ListedColormap

ROUTE_C, DETOUR_C, EXPL_C, WALL_C, ALERT_C, GOOD_C, BAD_C = "#E4533A", "#2563EB", "#FCEBE5", "#CBD4DC", "#F5B43C", "#1E8E61", "#C93A3A"


def plot_map(ax, log=None, route=None, detour=None, trip=None, title="", alert=False,
             explored_cells=None, explored_color=EXPL_C, bridge_blocked=False, show_ghf=True,
             route_label="shortest route (A*)"):
    """Draw the 6x6 map.
    log            A* expansion log; explored cells show g, h and f
    explored_cells any list of cells to shade (used for BFS)
    route          planned route (red), faded when a detour or trip is drawn
    detour / trip  path actually taken (blue)
    """
    ax.set_xlim(-0.5, world.COLS - 0.5); ax.set_ylim(world.ROWS - 0.5, -0.5)
    ax.set_xticks(range(world.COLS)); ax.set_yticks(range(world.ROWS))
    ax.set_xlabel("column"); ax.set_ylabel("row"); ax.set_aspect("equal")
    ax.tick_params(length=0)
    for s in ax.spines.values():
        s.set_visible(False)
    explored = {cell: (g, h, f) for cell, g, h, f in (log or [])}
    shaded = set(explored) | set(explored_cells or [])
    for r in range(world.ROWS):
        for c in range(world.COLS):
            cell = (r, c)
            face, edge, lw = "#F8FAFB", "#D6DEE6", 1
            if cell in world.WATER:
                face, edge = "#CFE3F5", "#CFE3F5"
            elif cell in world.WALLS:
                face = WALL_C
            elif cell == world.ALERT_CELL and bridge_blocked:
                face, edge, lw = "#F9DEDE", BAD_C, 2
            elif cell == world.ALERT_CELL and alert:
                face, edge, lw = "#FDF0D5", ALERT_C, 2
            elif cell in shaded:
                face = explored_color
            ax.add_patch(FancyBboxPatch((c - 0.45, r - 0.45), 0.9, 0.9,
                                        boxstyle="round,pad=0,rounding_size=0.12",
                                        facecolor=face, edgecolor=edge, linewidth=lw))
            if cell in world.WATER:
                ax.text(c, r, "≈", ha="center", va="center", fontsize=12, color="#8DB6DD")
            elif cell in world.WALLS:
                ax.text(c, r, "closed", ha="center", va="center", fontsize=8, color="#5A6A7B")
            elif show_ghf and cell in explored and cell not in (world.START, world.GOAL):
                g, h, f = explored[cell]
                ax.text(c - 0.38, r - 0.28, "g%d h%d" % (g, h), fontsize=7, color="#5A6A7B", family="monospace")
                ax.text(c + 0.38, r + 0.36, "f%d" % f, fontsize=7.5, ha="right", family="monospace")
    taken = trip or detour
    if route:
        ax.plot([c for r, c in route], [r for r, c in route], color=ROUTE_C, lw=4, solid_capstyle="round",
                alpha=0.35 if taken else 1, label=route_label, zorder=3)
    if taken:
        ax.plot([c for r, c in taken], [r for r, c in taken], color=DETOUR_C, lw=4, solid_capstyle="round",
                label="route actually driven", zorder=4)
    if bridge_blocked:
        r, c = world.ALERT_CELL
        ax.plot([c - 0.25, c + 0.25], [r - 0.25, r + 0.25], color=BAD_C, lw=4, zorder=6)
        ax.plot([c - 0.25, c + 0.25], [r + 0.25, r - 0.25], color=BAD_C, lw=4, zorder=6)
    for cell, label, color in ((world.START, "S", "#13202E"), (world.GOAL, "C", ROUTE_C)):
        ax.text(cell[1], cell[0], label, ha="center", va="center", fontsize=13, weight="bold", color="white",
                bbox=dict(boxstyle="round,pad=0.3", fc=color), zorder=7)
    r, c = world.ALERT_CELL
    ax.text(c - 0.38, r + 0.36, "bridge", ha="left", fontsize=7.5, color="#13202E", zorder=8)
    if route or taken:
        ax.legend(loc="upper right", fontsize=8, frameon=False, bbox_to_anchor=(1, -0.10), ncol=2)
    ax.set_title(title, loc="left", fontsize=11, weight="bold")


def plot_bayes(ax, prior, posterior, threshold_value):
    bars = ax.barh(["after report", "before report"], [posterior, prior], color=[ALERT_C, "#9AA8B6"], height=0.5)
    ax.axvline(threshold_value, color=DETOUR_C, lw=2, ls="--")
    ax.text(threshold_value + 0.01, 0.5, "MDP reroutes above %.1f%%" % (threshold_value * 100), color=DETOUR_C, fontsize=9)
    for b, v in zip(bars, [posterior, prior]):
        ax.text(v + 0.01, b.get_y() + b.get_height() / 2, "%.1f%%" % (v * 100), va="center", fontsize=10)
    ax.set_xlim(0, 1); ax.set_xlabel("P(bridge blocked)")
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    ax.set_title("Bayes: probability the bridge is blocked", loc="left", fontsize=11, weight="bold")


def plot_mdp(ax, ev, action):
    names = list(ev)
    for i, n in enumerate(names):
        ax.barh(i, ev[n], color=GOOD_C if n == "Continue" else DETOUR_C, alpha=1 if n == action else 0.4, height=0.5)
        ax.text(ev[n] + 0.2 if ev[n] >= 0 else 0.2, i, "%.2f" % ev[n], va="center", fontsize=10)
    ax.set_yticks(range(len(names))); ax.set_yticklabels(["EV(%s)" % n for n in names])
    ax.axvline(0, color="#5A6A7B", lw=0.8)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    ax.set_xlabel("expected value after one Bellman update")
    ax.set_title("MDP: decision = %s" % action.upper(), loc="left", fontsize=11, weight="bold")


def plot_policy_map(ax, points):
    """points = [(label, belief, extra), ...] drawn as dots on the map."""
    beliefs, grid = policy_grid()
    data = [[1 if grid[(e, b)] == "Reroute" else 0 for b in beliefs] for e in range(11)]
    ax.imshow(data, origin="lower", aspect="auto", extent=(-0.025, 1.025, -0.5, 10.5),
              cmap=ListedColormap(["#DAF1E7", "#DCE7FD"]))
    for label, b, e in points:
        ax.plot(b, e, "o", ms=11, mfc=ROUTE_C, mec="white", mew=2)
        ax.annotate(label, (b, e), xytext=(8, 6), textcoords="offset points", fontsize=9, weight="bold")
    ax.text(0.03, 9.5, "CONTINUE", color=GOOD_C, weight="bold")
    ax.text(0.97, 0.3, "REROUTE", color=DETOUR_C, weight="bold", ha="right")
    ax.set_xlabel("P(bridge blocked) after the report"); ax.set_ylabel("extra blocks of detour")
    ax.set_yticks(range(0, 11, 2))
    ax.set_title("Decision map: MDP choice for every risk and detour length", loc="left", fontsize=11, weight="bold")


def plot_evaluation(ax, res):
    names = list(res)
    vals = [res[n]["expected"] for n in names]
    cols = [ROUTE_C if n == "Assistant" else "#9AA8B6" for n in names]
    ax.barh(names[::-1], vals[::-1], color=cols[::-1], height=0.5)
    for i, n in enumerate(names[::-1]):
        r = res[n]
        ax.text(r["expected"] + 0.1, i, "%.2f   stuck %.1f%%, rerouted %.1f%%"
                % (r["expected"], 100 * r["stuck"], 100 * r["rerouted"]), va="center", fontsize=9)
    ax.set_xlim(0, max(vals) * 1.9)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    ax.set_xlabel("average reward per delivery")
    ax.set_title("Evaluation: 10,000 simulated deliveries", loc="left", fontsize=11, weight="bold")


## 1. A* search

**Input:** the map, restaurant S, customer C, 3 closed cells. **Heuristic:** Manhattan distance, admissible because every move costs at least 1.

Left: the cells A* expanded, with g, h and f. Right: the cells BFS expands to find a route of the same length.

In [ ]:
path, log = run_astar()
bfs_path, bfs_expanded = bfs(world.START, world.GOAL, world.WALLS)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 6.4))
plot_map(a1, log=log, route=path, title='A*: %d cells expanded' % len(log))
plot_map(a2, explored_cells=bfs_expanded, explored_color='#DCE7FD', route=bfs_path, show_ghf=False, route_label='shortest route (BFS)',
         title='BFS: %d cells expanded, same %d-block route' % (len(bfs_expanded), len(bfs_path) - 1))
plt.tight_layout(); plt.show()

## 2. Bayesian inference

**Input:** prior P(blocked) = 0.15, alert service with P(alert | blocked) = 0.85 and P(alert | clear) = 0.15, observation = ALERT.

Try `run_bayes('NO ALERT')` or `run_bayes(None)` to see the other cases.

In [ ]:
belief = run_bayes('ALERT')

## 3. MDP decision

**Input:** two road states (Clear, Blocked), two actions (Continue, Reroute), transition table, rewards. The reroute reward comes from the A* detour length. The printout also checks that full value iteration gives the same policy as one Bellman update.

In [ ]:
detour, _ = astar(world.START, world.GOAL, world.WALLS | {world.ALERT_CELL})
action, Q, ev, t = run_mdp(belief, len(path) - 1, len(detour) - 1)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.2))
plot_bayes(a1, PRIOR_BLOCKED, belief, t)
plot_mdp(a2, ev, action)
plt.tight_layout(); plt.show()

## 4. Evaluation

Three policies over 10,000 simulated evenings: always continue, always reroute, and the assistant (Bayes + MDP). Exact expected values are computed alongside the simulation.

In [ ]:
res = run_evaluation(Q, len(detour) - len(path))

fig, ax = plt.subplots(figsize=(10, 2.8))
plot_evaluation(ax, res)
plt.tight_layout(); plt.show()

## 5. The alert arrives on the way

The courier has already driven to (2, 1), one block before the bridge. A* replans from there, so the detour is longer and the bar for turning back is higher.

In [ ]:
d_way = run_on_the_way('ALERT')

## 6. Decision map

The MDP decision for every combination of risk and detour length. The two dots are the two alert timings.

In [ ]:
d_depart = decide_at(world.START, len(path) - 1, 'ALERT')
run_policy_map(d_depart['extra'], d_depart['belief'])

fig, ax = plt.subplots(figsize=(8, 4.4))
plot_policy_map(ax, [('alert before departure', d_depart['belief'], d_depart['extra']),
                     ('alert on the way', d_way['belief'], d_way['extra'])])
plt.tight_layout(); plt.show()

## 7. What really happens on the road

The assistant never sees whether the bridge is really blocked. `simulate_trip` plays the delivery out for a given truth. Left: the alert said nothing, the courier continued, and the bridge turned out to be blocked, so A* replans from (2, 1). Right: the alert fired, the courier rerouted before departure.

In [ ]:
unlucky = simulate_trip('NO ALERT', on_the_way=False, truth='blocked')
rerouted = simulate_trip('ALERT', on_the_way=False, truth='clear')
for name, t in (('no alert, bridge blocked', unlucky), ('alert, rerouted', rerouted)):
    print('%-26s %-9s %2d min  %s' % (name, t['action'], t['minutes'], ' | '.join(t['events'])))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 6.4))
plot_map(a1, route=path, trip=unlucky['trip'], bridge_blocked=True, title='Continued, bridge blocked: %d min' % unlucky['minutes'])
plot_map(a2, route=path, trip=rerouted['trip'], alert=True, title='Rerouted before departure: %d min' % rerouted['minutes'])
plt.tight_layout(); plt.show()

## 8. Bigger maps

The brief's map is 6×6. Two larger maps with a river and several bridges show how the same three algorithms scale. `use_map('10x10')` or `use_map('14x14')` switches every global the algorithms read.

On 10×10 an alert before departure means reroute (+2 blocks), but on the way the detour costs +12, so the courier continues. On 14×14 A* expands 37 cells where BFS needs 140.

In [ ]:
rows = []
for name in MAPS:
    use_map(name)
    path, log = astar(world.START, world.GOAL, world.WALLS)
    _, bexp = bfs(world.START, world.GOAL, world.WALLS)
    i = path.index(world.ALERT_CELL) - 1
    d0 = decide_at(world.START, len(path) - 1, 'ALERT')
    dw = decide_at(path[i], len(path) - 1 - i, 'ALERT')
    rows.append((name, len(path) - 1, len(log), len(bexp), d0['extra'], d0['action'], dw['extra'], dw['action']))
print('%-6s %5s %5s %5s   %-17s %-17s' % ('map', 'route', 'A*', 'BFS', 'alert at start', 'alert on the way'))
for n, rl, a, bb, e0, a0, ew, aw in rows:
    print('%-6s %5d %5d %5d   +%-2d %-13s +%-2d %-13s' % (n, rl, a, bb, e0, a0, ew, aw))

use_map('14x14')
path, log = astar(world.START, world.GOAL, world.WALLS)
bfs_path, bfs_expanded = bfs(world.START, world.GOAL, world.WALLS)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 7.6))
plot_map(a1, log=log, route=path, show_ghf=False, title='14×14, A*: %d cells expanded' % len(log))
plot_map(a2, explored_cells=bfs_expanded, explored_color='#DCE7FD', route=bfs_path, show_ghf=False, route_label='shortest route (BFS)',
         title='14×14, BFS: %d cells expanded' % len(bfs_expanded))
plt.tight_layout(); plt.show()
use_map('6x6')

## Test cases

52 tests: the algorithms are checked with independent methods (BFS, exact fractions, hand-computed Q values, full value iteration), and the command line is run on every map and scenario.

In [ ]:
import subprocess
print(subprocess.run([sys.executable, 'run_tests.py'], capture_output=True, text=True).stdout[-600:])